# AI应用基础 · 工单M4.2《工业智能决策助手系统》
**第11周 | 2课时(80分钟) | 理实一体**  | 教师参考版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 实现多轮对话记忆（历史拼接），测试连续追问的上下文保持；示范讲解 |
| **B 组长** | 标准达成 | ≥10问+RAG提升明显+多轮对话+容错通过 |
| **C 操作员** | 基础完成 | 正确回答≥5个典型问题 |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）

## 一、课前资讯（Predict·预测）
> **预习快问**：RAG相比纯LLM回答的优势是什么？企业数据合规要注意什么？

> **课前检查**：导学单+预习自测已完成；data/数据已就位、starter可运行（--check环境OK）

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m4kb"], check=True)
print("数据已生成: data/")

工作目录: /Users/joelgong/PycharmProjects/ai-course-teaching-resources/01-AI应用基础


M4.2 -> data/knowledge_base.json 13 条知识
完成。数据输出目录: /Users/joelgong/PycharmProjects/ai-course-teaching-resources/01-AI应用基础/data
数据已生成: data/


## 三、任务要求（Make）
**实施步骤**
1. 同一问题3版Prompt对比(朴素/角色/示例)
2. KnowledgeBase检索注入CONTEXT再生成
3. Agent: 工具判断→阈值判定→汇总回答
4. 容错: 超时重试+降级话术实测

**验收标准**：及格(D)=`正确回答≥5个典型问题` ；优秀(B+)=`≥10问+RAG提升明显+多轮对话+容错通过`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [2]:
# -*- coding: utf-8 -*-
"""M4.2 工业智能决策助手（本地模拟LLM + RAG + Agent）—— 学生代码框架
服务端(教师机): python mock_llm_server.py
本框架默认连 127.0.0.1:8000, 按机房实际IP修改 BASE_URL。
"""
import json
import re

import requests

BASE_URL = "http://127.0.0.1:8000"
KB_PATH = "data/knowledge_base.json"

### Starter 2：`def chat(messages, retries=2, timeout=5):`
（Investigate: 读代码→Modify: 补全TODO）

In [3]:
def chat(msgs):
    """调用本地LLM(OpenAI兼容), 带重试与降级(容错必做)"""
    import time
    import requests
    BASE = "http://127.0.0.1:8000"
    last = None
    for i in range(3):
        try:
            r = requests.post(f"{BASE}/v1/chat/completions",
                              json={"messages": msgs}, timeout=5)
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"]
        except requests.RequestException as e:
            last = e
            time.sleep(1 + i)
    return f"[降级] LLM服务不可达，已回退规则通道（{last}）"

### 💡 分层提示（先自己试5分钟再看）
**思路提示**
> - （在此处编写你的代码——依据上方注释/文档字符串）
> - （在此处编写你的代码——依据上方注释/文档字符串）
> - TODO-4: RAG对比: 检索结果注入后再问, 记录是否更"有依据"
>   TODO-5: Agent全流程问答记录表(问题/回答/依据)

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位TODO行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改参数/换实现对比效果，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方"验收标准"，及格(D)=`正确回答≥5个典型问题`，优秀(B+)=`≥10问+RAG提升明显+多轮对话+容错通过`

### Starter 3：`class KnowledgeBase:`
（Investigate: 读代码→Modify: 补全TODO）

In [4]:
class KnowledgeBase:
    """TODO-2 完成态：知识库检索用二元词重合打分，按类目取 topk（选型对照见工程化项目 m42_ops_rag 的决策矩阵）"""

    def __init__(self, path=KB_PATH):
        with open(path, encoding="utf-8") as f:
            self.kb = json.load(f)
        self.docs = [t for items in self.kb.values() for t in items]

    def retrieve(self, query, topk=2):
        """返回最相关的topk条知识(字符串列表)"""
        # 【参考答案】二元词重合检索(与工程化项目 m42_ops_rag retrieve 同源)
        def bigrams(text):
            out = set()
            for seg in re.findall(r"[\u4e00-\u9fff]+", str(text)):
                if len(seg) < 2:
                    continue
                out |= {seg[i:i + 2] for i in range(len(seg) - 1)}
                out.add(seg)
            return out

        q = bigrams(query)
        if not q:
            return self.docs[:topk]
        scored = []
        for t in self.docs:
            k = bigrams(t)
            score = len(q & k) / len(q)
            if score > 0:
                scored.append((score, t))
        scored.sort(key=lambda x: -x[0])
        return [t for _, t in scored[:topk]] or self.docs[:topk]


### Starter 4：`class Tool:`
（Investigate: 读代码→Modify: 补全TODO）

In [5]:
class Tool:
    """工具: 名称+功能函数"""

    def __init__(self, name, func):
        self.name, self.func = name, func

    def run(self, *a):
        return self.func(*a)

### Starter 5：`class Agent:`
（Investigate: 读代码→Modify: 补全TODO）

In [6]:
class Agent:
    """TODO-3 完成态：两步推理——先按阈值判定工具判超标，再检索知识库组装 Prompt 生成回答"""

    def __init__(self, kb):
        self.kb = kb
        self.tools = {
            "查询知识库": Tool("查询知识库", kb.retrieve),
            "阈值判定": Tool("阈值判定",
                          lambda v: "超标,须停机检查" if float(v) > 4.5
                          else "正常范围"),
        }

    def ask(self, question):
        # 【参考答案】步骤1: 含振动数值(mm/s)→先走阈值判定工具, 再检索依据
        m = re.search(r"([\d.]+)\s*mm/s", question)
        hits = self.kb.retrieve(question, topk=2)
        basis = "；".join(hits) if hits else "知识库无命中"
        if m:
            verdict = self.tools["阈值判定"].run(m.group(1))
            messages = [
                {"role": "system",
                 "content": "你是设备运维助手: 只依据给定依据回答; 依据不足则转人工工单。"},
                {"role": "user",
                 "content": f"振动值{m.group(1)}mm/s 判定: {verdict}\n依据: {basis}"},
            ]
            return {"answer": chat(messages), "basis": basis,
                    "grounded": bool(hits)}
        # 【参考答案】步骤2: 非数值问题→检索注入后回答; 无命中则拒答(防幻觉)
        if not hits:
            return {"answer": "查无依据: 请补充设备编号与测点数据后再问(宁说无据, 不硬编)。",
                    "basis": "", "grounded": False}
        messages = [
            {"role": "system",
             "content": "你是设备运维助手: 只依据【依据】内容回答; 依据不足时明确说明并转人工工单。"},
            {"role": "user",
             "content": f"问题: {question}\n【依据】\n{basis}"},
        ]
        return {"answer": chat(messages), "basis": basis, "grounded": True}


### Starter 6：`PROMPTS = {`
（Investigate: 读代码→Modify: 补全TODO）

In [7]:
PROMPTS = {
    "朴素": "回答问题: {q}",
    "角色设定": "你是资深设备工程师, 按规范回答: {q}",
    "带示例": "示例问: 空压机排气压力低怎么办?\n示例答: 先查气源泄漏…\n现在回答: {q}",
}

### Starter 7：`def main():`
（Investigate: 读代码→Modify: 补全TODO）

In [8]:
def main():
    kb = KnowledgeBase()
    agent = Agent(kb)
    questions = [
        "泵振动7.5mm/s怎么处理？",
        "轴承温度82℃要不要停机？",
        "变频器过流报警如何排查？",
        "液压油污染度NAS 10级怎么办？",
        "PLC从站通信中断排查顺序？",
    ]
    # ---- 1) Prompt对比(同一问题三版) ----
    for name, tpl in PROMPTS.items():
        msg = [{"role": "user", "content": tpl.format(q=questions[0])}]
        print(f"[{name}] {chat(msg)[:60]}...\n")
    # 【参考答案】TODO-4 RAG对比: 裸问 vs 检索注入后再问, 记录依据性差异
    q0 = questions[2]
    hits0 = kb.retrieve(q0, topk=2)
    print("== 裸问 ==\n", chat([{"role": "user", "content": q0}])[:80])
    rag_msg = [
        {"role": "system", "content": "只依据【依据】回答, 不足则说明。"},
        {"role": "user", "content": f"{q0}\n【依据】\n" + "\n".join(hits0)},
    ]
    print("== RAG注入后 ==\n", chat(rag_msg)[:80], "\n依据:", hits0)
    # 【参考答案】TODO-5 Agent全流程问答记录表(问题/回答/依据/是否grounded)
    print(f"{'问题':<18} {'grounded':<8} 回答摘要")
    for q in questions:
        r = agent.ask(q)
        ans = r["answer"] if isinstance(r, dict) else str(r)
        grounded = r.get("grounded", "-") if isinstance(r, dict) else "-"
        print(f"{q:<18} {str(grounded):<8} {ans[:48]}...")


### Starter 8：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [9]:
if __name__ == "__main__":
    main()

[朴素] [降级] LLM服务不可达，已回退规则通道（HTTPConnectionPool(host='127.0.0.1', p...



[角色设定] [降级] LLM服务不可达，已回退规则通道（HTTPConnectionPool(host='127.0.0.1', p...



[带示例] [降级] LLM服务不可达，已回退规则通道（HTTPConnectionPool(host='127.0.0.1', p...



== 裸问 ==
 [降级] LLM服务不可达，已回退规则通道（HTTPConnectionPool(host='127.0.0.1', port=8000): Max retri


== RAG注入后 ==
 [降级] LLM服务不可达，已回退规则通道（HTTPConnectionPool(host='127.0.0.1', port=8000): Max retri 
依据: ['变频器过流报警时，应先检查负载卡滞与加减速时间设置，再检查电流检测板。', '轴承温度报警阈值设定为75℃，达到85℃应立即停机并检查润滑与冷却回路。']
问题                 grounded 回答摘要


泵振动7.5mm/s怎么处理？    True     [降级] LLM服务不可达，已回退规则通道（HTTPConnectionPool(host='1...


轴承温度82℃要不要停机？      True     [降级] LLM服务不可达，已回退规则通道（HTTPConnectionPool(host='1...


变频器过流报警如何排查？       True     [降级] LLM服务不可达，已回退规则通道（HTTPConnectionPool(host='1...


液压油污染度NAS 10级怎么办？  True     [降级] LLM服务不可达，已回退规则通道（HTTPConnectionPool(host='1...


PLC从站通信中断排查顺序？     True     [降级] LLM服务不可达，已回退规则通道（HTTPConnectionPool(host='1...


## 四、提交要求与避坑
- 提交: `学号_姓名_M4.2.py` 或本notebook(含输出)
- ⚠ requests必须timeout=5并捕获异常做降级
- ⚠ 企业数据不出域：课内用本地模拟LLM，真实API选做且先脱敏
- ⚠ 检索只取Top2，Prompt过长反而稀释注意力

---

## 教师参考答案（来源：`教师参考/solutions.py` · `m42()`）

> **发放纪律：本附录勿随学生版下发。**

上方学生版 TODO 与下方参考实现的对应关系：

| 学生版函数 | 说明 |
|---|---|
| `KnowledgeBase()` | 见下方 `m42()` 内对应逻辑 |
| `Agent()` | 见下方 `m42()` 内对应逻辑 |
| `main()` | 见下方 `m42()` 内对应逻辑 |

In [10]:
def m42():
    import re
    import time
    import requests

    BASE = "http://127.0.0.1:8000"

    def chat(msgs):
        last = None
        for i in range(3):                          # 重试: 覆盖服务冷启动窗口
            try:
                r = requests.post(f"{BASE}/v1/chat/completions",
                                  json={"messages": msgs}, timeout=5)
                r.raise_for_status()
                return r.json()["choices"][0]["message"]["content"]
            except requests.RequestException as e:
                last = e
                time.sleep(1 + i)
        raise last

    kb = json.load(open("data/knowledge_base.json", encoding="utf-8"))
    docs = [t for v in kb.values() for t in v]

    def retrieve(q, topk=2):
        qs = set(re.findall(r"[\u4e00-\u9fa5a-zA-Z0-9]+", q))
        return sorted(docs, key=lambda t: -len(qs & set(
            re.findall(r"[\u4e00-\u9fa5a-zA-Z0-9]+", t))))[:topk]

    qs = ["泵振动7.5mm/s怎么处理？", "轴承温度82℃要不要停机？",
          "变频器过流报警如何排查？"]
    for q in qs:
        ctx = "\n".join(retrieve(q))
        a = chat([{"role": "system", "content": "CONTEXT:" + ctx},
                  {"role": "user", "content": q}])
        print("Q:", q, "\nA:", a[:90], "\n")
    print("Agent演示: 振动7.5>4.5阈值 → 判定超标 → 检索知识库 → 生成停机检查建议。")

## 运行与验证

上方参考实现可直接运行（合成数据、CPU 离线、结果可复现）。
多解法对比见《多解法手册.md》。